# Portfolio Stress Lab: Paper Walkthrough

This notebook presents the project as a compact research paper. It is designed to make the workflow easy to read: first the question, then the data, then the method, then the results.

The notebook is meant to be read top-to-bottom like a memo or technical note.

## Abstract

This project accepts a portfolio file and historical returns, runs stress scenarios and produces a research-style risk report. The emphasis is on transparency: a user should be able to replace the sample CSVs with their own inputs and reproduce the analysis without modifying the code.

## 1. Problem Statement

The objective is to measure portfolio vulnerability under adverse market moves. A single risk metric is insufficient, so the analysis combines base risk, scenario stress, tail risk, attribution, concentration and reverse stress checks.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'stresslab').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')

ROOT = find_repo_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from stresslab import StressTestEngine

DATA = ROOT / 'data'
SCENARIOS = ROOT / 'scenarios'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)

returns = pd.read_csv(DATA / 'sample_returns.csv', index_col=0, parse_dates=True)
portfolio = pd.read_csv(DATA / 'sample_portfolio.csv')

display(Markdown(f'**Repository root:** `{ROOT}`'))
display(Markdown(f'**Portfolio rows:** {len(portfolio)}'))
display(Markdown(f'**Return observations:** {len(returns)}'))

## 2. Methodology

The engine first loads and validates the portfolio, then aligns the return matrix, computes base risk metrics, and finally evaluates the configured stress scenarios. This sequence is intentionally simple so that the analysis is easy to audit.

In [ ]:
engine = StressTestEngine(returns=returns, portfolio=portfolio, confidence_level=0.99)
report = engine.run_all(scenarios=SCENARIOS)

print(report.summary())

## 3. Results

The tables below show the main outputs: base risk, scenario losses, risk contributions and sector exposure. These are the core outputs that make the project look like a research note instead of a generic script.

In [ ]:
display(Markdown('### Base risk and scenario losses'))
display(report.scenario_results[['direct_loss', 'vol_corr_loss_proxy', 'total_loss']].sort_values('total_loss').style.format('{:.2%}'))
display(Markdown('### Risk contributions'))
display(report.risk_contributions.head(10).to_frame('risk_contribution').style.format('{:.4f}'))
display(Markdown('### Sector exposure'))
display(report.sector_exposure.to_frame('weight').style.format('{:.2%}'))

## 4. Interpretation

The main things to look at are:

- which scenario produces the worst estimated loss,
- which assets contribute most to risk,
- whether a small set of sectors dominates the portfolio,
- and whether the reverse-stress thresholds are close to current risk.

That makes the notebook useful both as an analysis artifact and as a presentation piece.

## 5. Limitations

The outputs are research estimates based on historical data and stylized shocks. They should be treated as a structured stress-testing framework rather than a production risk engine.